In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows, find_peaks
from ipywidgets import RadioButtons, VBox, HBox, interactive_output
from IPython.display import display

def analyze_window(window_type):
    plt.close('all')

    # ------------------------------------------------------------
    # Configuration
    # ------------------------------------------------------------
    N = 128
    alpha_val = 2.0
    n_fft = 16384

    # ------------------------------------------------------------
    # Window generation
    # DFT-even windows are used for comparison with Harris.
    # ------------------------------------------------------------
    if window_type == 'Rectangular':
        w = windows.boxcar(N)
    elif window_type == 'Bartlett':
        w = windows.bartlett(N, sym=False)
    elif window_type == 'Hann':
        w = windows.hann(N, sym=False)
    elif window_type == 'Hamming':
        w = windows.hamming(N, sym=False)
    elif window_type == 'Blackman':
        w = windows.blackman(N, sym=False)
    elif window_type == 'Kaiser':
        w = windows.kaiser(N, beta=np.pi * alpha_val)
    else:
        w = windows.boxcar(N)

    # ------------------------------------------------------------
    # Coherent Gain and Equivalent Noise Bandwidth
    # ------------------------------------------------------------
    sum_w = np.sum(w)
    sum_w2 = np.sum(w**2)

    g_coh = sum_w / N
    b_eq = N * sum_w2 / sum_w**2

    # ------------------------------------------------------------
    # Scalloping Loss
    # Worst case occurs at a half-bin frequency displacement.
    # ------------------------------------------------------------
    n = np.arange(N)
    half_bin_gain = np.abs(np.sum(w * np.exp(-1j * np.pi * n / N))) / sum_w
    scallop_loss_db = -20.0 * np.log10(half_bin_gain)

    # ------------------------------------------------------------
    # Harris Worst-Case Processing Loss
    # ------------------------------------------------------------
    window_processing_loss_db = 10.0 * np.log10(b_eq)
    processing_loss_db = window_processing_loss_db + scallop_loss_db

    # ------------------------------------------------------------
    # High-resolution spectrum
    # ------------------------------------------------------------
    w_padded = np.zeros(n_fft, dtype=complex)
    w_padded[:N] = w

    W_spec = np.fft.fftshift(np.fft.fft(w_padded, n_fft))
    W_mag = np.abs(W_spec)
    W_mag /= np.max(W_mag)

    W_mag_dB = 20.0 * np.log10(np.maximum(W_mag, 1e-12))

    freqs_bins = np.fft.fftshift(np.fft.fftfreq(n_fft, d=1 / N))

    # ------------------------------------------------------------
    # Locate the first null after the main lobe
    # ------------------------------------------------------------
    center_idx = n_fft // 2

    positive_mag = W_mag[center_idx:]

    minima, _ = find_peaks(-positive_mag, prominence=1e-6)

    if len(minima) > 0:
        first_null_idx = center_idx + minima[0] + 1
    else:
        first_null_idx = center_idx + 1

    # ------------------------------------------------------------
    # Locate the highest sidelobe after the first null
    # ------------------------------------------------------------
    sidelobe_region = W_mag[first_null_idx:]

    peaks, _ = find_peaks(sidelobe_region, prominence=1e-6)

    if len(peaks) > 0:
        sidelobe_indices = first_null_idx + peaks
        highest_sidelobe_idx = sidelobe_indices[np.argmax(W_mag[sidelobe_indices])]
        max_sidelobe_dB = W_mag_dB[highest_sidelobe_idx]
    else:
        highest_sidelobe_idx = None
        max_sidelobe_dB = np.nan

    # ------------------------------------------------------------
    # Plotting
    # ------------------------------------------------------------
    fig = plt.figure(figsize=(14, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.2, 1], wspace=0.3)

    # Spectrum
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(freqs_bins, W_mag_dB, 'b-', lw=1.5)

    title_str = f"Spectrum of {window_type} (N={N})"

    if window_type == 'Kaiser':
        title_str += f", α={alpha_val:.1f}"

    ax1.set_title(title_str, fontsize=11, fontweight='bold')
    ax1.set_xlabel("Frequency [Bins]", fontsize=9)
    ax1.set_ylabel("Magnitude [dB]", fontsize=9)
    ax1.set_ylim(-100, 5)
    ax1.set_xlim(-8, 8)
    ax1.grid(True, linestyle='--', alpha=0.6)

    # Mark highest sidelobe
    if highest_sidelobe_idx is not None:
        ax1.plot(freqs_bins[highest_sidelobe_idx], max_sidelobe_dB, 'ro', markersize=5)

    # Metrics bar chart
    ax2 = fig.add_subplot(gs[0, 1])

    metrics_names = ['B_eq\n(bins)', 'G_coh', 'H_side\n(-dB/10)']
    metrics_values = [b_eq, g_coh, abs(max_sidelobe_dB) / 10]

    ax2.bar(metrics_names, metrics_values, color=['#1f77b4', '#2ca02c', '#d62728'], alpha=0.85)

    ax2.set_title("Window Performance Metrics", fontsize=11, fontweight='bold')
    ax2.set_ylabel("Value / Scale", fontsize=9)
    ax2.grid(True, axis='y', linestyle='--', alpha=0.6)

    y_max = max(metrics_values)
    ax2.set_ylim(0, y_max * 1.15)

    ax2.text(0, b_eq + 0.05, f"{b_eq:.3f}", ha='center', fontsize=9, fontweight='bold')
    ax2.text(1, g_coh + 0.05, f"{g_coh:.3f}", ha='center', fontsize=9, fontweight='bold')
    ax2.text(2, abs(max_sidelobe_dB) / 10 + 0.05, f"{max_sidelobe_dB:.1f} dB", ha='center', fontsize=9, fontweight='bold')

    plt.show()

    # ------------------------------------------------------------
    # Text output
    # ------------------------------------------------------------
    print("=" * 75)
    print(f" WINDOW PERFORMANCE METRICS: {window_type.upper()} (N = {N})" + (f", α = {alpha_val:.1f}" if window_type == 'Kaiser' else ""))
    print("=" * 75)
    print(f"• Coherent Gain (G_coh)             : {g_coh:.4f}")
    print(f"• Equivalent Noise Bandwidth (B_eq) : {b_eq:.4f} bins")
    print(f"• Scalloping Loss                   : {scallop_loss_db:.4f} dB")
    print(f"• Processing Loss (L_p)             : {processing_loss_db:.4f} dB")
    print(f"• Highest Sidelobe Level (H_side)   : {max_sidelobe_dB:.2f} dB")
    print("=" * 75)

# ------------------------------------------------------------
# Widget Setup
# ------------------------------------------------------------
window_radio = RadioButtons(
    options=['Rectangular', 'Bartlett', 'Hann', 'Hamming', 'Blackman', 'Kaiser'],
    value='Rectangular',
    description='Window:',
    disabled=False
)

controls = VBox([window_radio])
out = interactive_output(analyze_window, {'window_type': window_radio})

display(HBox([controls, out]))